In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import numpy as np
import polars as pl
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

import importlib
from matplotlib_inline.backend_inline import set_matplotlib_formats
import mercury_runs.analysis.study_dataset as sd
import scripts.analysis.load_study_dataset as loader_mod
import mercury_runs.analysis.plotting as plotting

importlib.reload(sd)
importlib.reload(loader_mod)
importlib.reload(plotting)
%matplotlib inline
set_matplotlib_formats("svg")
matplotlib.rcParams["savefig.transparent"] = False
matplotlib.rcParams["figure.facecolor"] = "white"
matplotlib.rcParams["axes.facecolor"] = "white"
plotting.apply_light_mode_plot_style()

In [ ]:
REPO_ROOT = Path("/Users/lcarv/PycharmProjects/mercuryv5")
study_root = (REPO_ROOT / "results" / "current" / "studies" /
              "levels_13_21_22_memory_study_new")

print(study_root)
print(study_root.exists())

dataset = loader_mod.load_mercury_study_dataset(study_root)

In [ ]:
from mercury_runs.save_results import read_bundle_meta


def _xlogx(x: float) -> float:
    x = float(x)
    return 0.0 if x <= 0.0 else x * np.log(x)


def behaviour_policy_entropy_nats(p_coll: float, epsilon: float, action_count: int) -> float:
    action_count = int(action_count)
    epsilon = float(epsilon)

    p_star = 1.0 - epsilon + epsilon / action_count
    p_other = epsilon / action_count

    h_no_collision = -_xlogx(p_star) - (action_count - 1) * _xlogx(p_other)

    return float(p_coll) * np.log(action_count) + (1.0 - float(p_coll)) * h_no_collision


def repeat_action_probability(p_coll: float, epsilon: float, action_count: int) -> float:
    return (
        float(p_coll) / action_count
        + (1.0 - float(p_coll)) * (1.0 - float(epsilon) + float(epsilon) / action_count)
    )


def _resolve_source_dataset_path(meta, datasets_root=Path("datasets")):
    dataset_name = meta.source.dataset_parquet_name
    if not dataset_name:
        return None

    matches = sorted(datasets_root.rglob(dataset_name))
    return matches[-1] if matches else None


def _action_count(meta, source_dataset_path):
    if source_dataset_path is not None:
        names = pl.scan_parquet(source_dataset_path).collect_schema().names()
    else:
        names = (meta.source_dataset_metadata or {}).get("table", {}).get("columns", [])

    return sum(str(name).startswith("action_") for name in names)


def _collision_rate_hat(meta, source_dataset_path):
    if source_dataset_path is not None:
        collisions = (
            pl.read_parquet(source_dataset_path, columns=["collision"])
            .get_column("collision")
            .to_numpy()
        )
        return float(np.asarray(collisions[:-1], dtype=bool).mean()), "source_parquet"

    source_metadata = meta.source_dataset_metadata or {}
    generation_stats = source_metadata.get("generation_stats", {}) or {}

    saved_steps = int(generation_stats.get("saved_steps") or source_metadata.get("num_steps"))
    if bool(source_metadata.get("valid_trajectories_only")):
        collision_count = 0
    else:
        collision_count = int(generation_stats["dropped_collision_steps"])

    return float(collision_count / max(saved_steps - 1, 1)), "embedded_metadata"


entropy_rows = []

for row in dataset.runs.select(
    "run_id",
    "level",
    "seed",
    "rand_prob",
    "num_steps",
    "mercury_valid_trajectories_only",
    "mercury_split_sensory_raw_latent_valid",
    "mercury_states_parquet",
).iter_rows(named=True):
    meta = read_bundle_meta(Path(row["mercury_states_parquet"]))
    if meta is None:
        raise ValueError(f"No embedded source metadata in {row['mercury_states_parquet']}")

    source_metadata = meta.source_dataset_metadata or {}
    source_dataset_path = _resolve_source_dataset_path(meta)
    action_count = _action_count(meta, source_dataset_path)

    epsilon = float(row["rand_prob"])
    raw_p_coll_hat, entropy_source = _collision_rate_hat(meta, source_dataset_path)

    latent_valid_only = bool(row["mercury_valid_trajectories_only"]) or bool(
        row["mercury_split_sensory_raw_latent_valid"]
    )
    latent_p_coll_hat = 0.0 if latent_valid_only else raw_p_coll_hat

    raw_h_nats = behaviour_policy_entropy_nats(
        p_coll=raw_p_coll_hat,
        epsilon=epsilon,
        action_count=action_count,
    )
    latent_h_nats = behaviour_policy_entropy_nats(
        p_coll=latent_p_coll_hat,
        epsilon=epsilon,
        action_count=action_count,
    )

    entropy_rows.append(
        {
            "run_id": row["run_id"],
            "source_dataset_run_id": source_metadata.get("run_id") or meta.source.dataset_run_id,
            "level": row["level"],
            "seed": row["seed"],
            "epsilon": epsilon,
            "rand_prob": epsilon,
            "num_steps": row["num_steps"],
            "action_count": action_count,
            "mercury_valid_trajectories_only": bool(row["mercury_valid_trajectories_only"]),
            "mercury_split_sensory_raw_latent_valid": bool(row["mercury_split_sensory_raw_latent_valid"]),
            "p_coll_hat": raw_p_coll_hat,
            "rho_bar_hat": (1.0 - epsilon) * (1.0 - raw_p_coll_hat),
            "repeat_action_probability_hat": repeat_action_probability(raw_p_coll_hat, epsilon, action_count),
            "policy_entropy_nats": raw_h_nats,
            "policy_entropy_bits": raw_h_nats / np.log(2.0),
            "latent_p_coll_hat": latent_p_coll_hat,
            "latent_rho_bar_hat": (1.0 - epsilon) * (1.0 - latent_p_coll_hat),
            "latent_repeat_action_probability_hat": repeat_action_probability(
                latent_p_coll_hat,
                epsilon,
                action_count,
            ),
            "latent_policy_entropy_nats": latent_h_nats,
            "latent_policy_entropy_bits": latent_h_nats / np.log(2.0),
            "latent_policy_equivalent_actions": 2**(latent_h_nats / np.log(2.0)),
            "entropy_source": entropy_source,

        }
    )

policy_entropy_by_run = pl.DataFrame(entropy_rows).sort(
    ["level", "rand_prob", "mercury_split_sensory_raw_latent_valid", "seed"]
)

policy_entropy_by_regime = (
    policy_entropy_by_run
    .unique(
        subset=[
            "source_dataset_run_id",
            "mercury_valid_trajectories_only",
            "mercury_split_sensory_raw_latent_valid",
        ],
        keep="first",
    )
    .group_by(["level", "rand_prob", "mercury_split_sensory_raw_latent_valid"])
    .agg(
        pl.len().alias("n_trajectories"),
        pl.col("action_count").first().alias("action_count"),
        pl.col("p_coll_hat").mean().alias("p_coll_hat_mean"),
        pl.col("p_coll_hat").std().fill_null(0.0).alias("p_coll_hat_std"),
        pl.col("rho_bar_hat").mean().alias("rho_bar_hat_mean"),
        pl.col("repeat_action_probability_hat").mean().alias("repeat_action_probability_hat_mean"),
        pl.col("policy_entropy_nats").mean().alias("policy_entropy_nats_mean"),
        pl.col("policy_entropy_nats").std().fill_null(0.0).alias("policy_entropy_nats_std"),
        pl.col("policy_entropy_bits").mean().alias("policy_entropy_bits_mean"),
        pl.col("policy_entropy_bits").std().fill_null(0.0).alias("policy_entropy_bits_std"),
        pl.col("latent_p_coll_hat").mean().alias("latent_p_coll_hat_mean"),
        pl.col("latent_p_coll_hat").std().fill_null(0.0).alias("latent_p_coll_hat_std"),
        pl.col("latent_rho_bar_hat").mean().alias("latent_rho_bar_hat_mean"),
        pl.col("latent_repeat_action_probability_hat").mean().alias("latent_repeat_action_probability_hat_mean"),
        pl.col("latent_policy_entropy_nats").mean().alias("latent_policy_entropy_nats_mean"),
        pl.col("latent_policy_entropy_nats").std().fill_null(0.0).alias("latent_policy_entropy_nats_std"),
        pl.col("latent_policy_entropy_bits").mean().alias("latent_policy_entropy_bits_mean"),
        pl.col("latent_policy_entropy_bits").std().fill_null(0.0).alias("latent_policy_entropy_bits_std"),
        pl.col("latent_policy_equivalent_actions").mean().fill_null(0.0).alias
        ("latent_policy_equivalent_actions"),
        pl.col("latent_policy_equivalent_actions").std().fill_null(0.0).alias
        ("latent_policy_equivalent_actions_std"),
        pl.col("entropy_source").unique().alias("entropy_sources"),
    )
    .sort(["level", "rand_prob", "mercury_split_sensory_raw_latent_valid"])
)

policy_entropy_by_regime


In [ ]:
print(sd.__file__)
print(dataset.runs.columns)
print(dataset.history.shape)
print(dataset.states.shape)
print(dataset.history.columns)

In [ ]:
results = dataset.history_with_run_metadata(["level", "rand_prob",
                                             "weight_memory", "mercury_split_sensory_raw_latent_valid"])
print(results.shape)


In [ ]:
def state_recall_from_cooccurrence(matrix):
    if matrix is None:
        return None
    if hasattr(matrix, "to_list"):
        matrix = matrix.to_list()
    C = np.asarray(matrix, dtype=np.float64)
    if C.ndim != 2 or C.size == 0:
        return None
    total = C.sum()
    if total <= 0:
        return None
    return float(C.max(axis=0).sum() / total)


history_with_state_recall = dataset.history.with_columns(
    pl.col("sensory_exact_state_cooccurrence_matrix")
    .map_elements(state_recall_from_cooccurrence, return_dtype=pl.Float64)
    .alias("sensory_state_recall"),
    pl.col("latent_exact_state_cooccurrence_matrix")
    .map_elements(state_recall_from_cooccurrence, return_dtype=pl.Float64)
    .alias("latent_state_recall"),
)

results = history_with_state_recall.join(
    dataset.run_metadata([
        "level",
        "rand_prob",
        "weight_memory",
        "mercury_split_sensory_raw_latent_valid",
    ]),
    on="run_id",
    how="left",
    coalesce=True,
)

print(results.select("sensory_state_recall", "latent_state_recall").drop_nulls().head())


In [ ]:

level_20_latent_precision =(
    results
    .filter(pl.col("level") == 21,
            pl.col("rand_prob") == 0.3,
            pl.col("weight_memory") == 0.8,
            pl.col("mercury_split_sensory_raw_latent_valid") == False,
            )
    .select(
        pl.col("run_id"),
        pl.col("step"),
        pl.col("latent_precision")
    )
)

runs = level_20_latent_precision.unique("run_id").select("run_id").to_series().to_list()

for run in runs:
    print(run)
    level_20_latent_precision_filtered = level_20_latent_precision.filter(pl.col("run_id") == run)


    x_values = level_20_latent_precision_filtered["step"].to_numpy()
    y_values = (level_20_latent_precision_filtered["latent_precision"]
                .to_numpy())
    print(level_20_latent_precision_filtered.shape)
    print(x_values[:5], y_values[:5])
    plt.plot(x_values, y_values)

plt.show()

In [ ]:

filtered = (
    results
    .filter(
        pl.col("level") == 21,
        pl.col("rand_prob") == 0.3,
        pl.col("weight_memory") == 0.8,
        pl.col("mercury_split_sensory_raw_latent_valid") == True,
    )
    .select(
        "run_id",
        "step",
        "latent_precision",
        "sensory_precision",  # <-- include this
    )
)

# Aggregate
summary = (
    filtered
    .group_by("step")
    .agg(
        pl.col("latent_precision").mean().alias("latent_mean"),
        pl.col("latent_precision").std().alias("latent_std"),

        pl.col("sensory_precision").mean().alias("sensory_mean"),
        pl.col("sensory_precision").std().alias("sensory_std"),
    )
    .sort("step")
)

x = summary["step"].to_numpy()

latent_mean = summary["latent_mean"].to_numpy()
latent_std = summary["latent_std"].fill_null(0).to_numpy()

sensory_mean = summary["sensory_mean"].to_numpy()
sensory_std = summary["sensory_std"].fill_null(0).to_numpy()

# Plot latent (mean + shaded std)
plt.plot(x, latent_mean, label="Latent precision")
plt.fill_between(
    x,
    latent_mean - latent_std,
    latent_mean + latent_std,
    alpha=0.3,
)

# Plot sensory baseline
plt.plot(
    x,
    sensory_mean,
    linestyle="--",
    label="Sensory precision (baseline)"
)

# Optional: add shading for sensory as well
# plt.fill_between(
#     x,
#     sensory_mean - sensory_std,
#     sensory_mean + sensory_std,
#     alpha=0.15,
# )

plt.xlabel("Step")
plt.ylabel("Precision")
plt.legend()
plt.show()

In [ ]:
comparison_plot_base = plotting.NotebookPlot(
    shared_filters={
        "level": 21,
        "rand_prob": 0.3,
    },
    series=[
        plotting.NotebookSeries(
            name="latent_precision_main",
            metric_column_name="latent_mean_total_variation",
            label="Latent transition precision",
            filter_values={
                "weight_memory": 1.0,
                "mercury_split_sensory_raw_latent_valid": True,
            },
            band_alpha=0.25,
        ),
        plotting.NotebookSeries(
            name="sensory_precision_baseline",
            metric_column_name="sensory_precision",
            label="Sensory precision",
            filter_values={
                "weight_memory": 0.8,
                "mercury_split_sensory_raw_latent_valid": False,
            },
            line_style="--",
            show_standard_deviation_band=False,
            is_baseline=True,
        ),
    ],
    y_axis_label="Precision",
    y_limits=(0.0, 1.0),
    use_grid=True,
)

comparison_plot = plotting.NotebookPlot(
    shared_filters=comparison_plot_base.shared_filters,
    series=comparison_plot_base.series,
    title_spec=plotting.prefilled_title_spec_from_canonical_parameters(
        comparison_plot_base,
        plotting.CANONICAL_TITLE_PARAMETERS,
        parameter_order=plotting.CANONICAL_TITLE_PARAMETER_ORDER,
        default_source_series_name="latent_precision_main",
        metric_label_mode="first_series",
    ),
    y_axis_label=comparison_plot_base.y_axis_label,
    y_limits=comparison_plot_base.y_limits,
    use_grid=comparison_plot_base.use_grid,
)

plotting.display_notebook_plot(results, comparison_plot)

In [ ]:
def with_auto_title(plot_base, metric_label_mode="first_series", default_source_series_name=None):
    return plotting.NotebookPlot(
        shared_filters=plot_base.shared_filters,
        series=plot_base.series,
        title_spec=plotting.prefilled_title_spec_from_canonical_parameters(
            plot_base,
            plotting.CANONICAL_TITLE_PARAMETERS,
            parameter_order=plotting.CANONICAL_TITLE_PARAMETER_ORDER,
            default_source_series_name=default_source_series_name,
            metric_label_mode=metric_label_mode,
        ),
        figure_size=plot_base.figure_size,
        x_column_name=plot_base.x_column_name,
        run_column_name=plot_base.run_column_name,
        x_axis_label=plot_base.x_axis_label,
        y_axis_label=plot_base.y_axis_label,
        y_limits=plot_base.y_limits,
        x_limits=plot_base.x_limits,
        show_legend=plot_base.show_legend,
        legend_location=plot_base.legend_location,
        use_grid=plot_base.use_grid,
        aggregation_mode=plot_base.aggregation_mode,
        mean_alias_suffix=plot_base.mean_alias_suffix,
        std_alias_suffix=plot_base.std_alias_suffix,
        fill_null_standard_deviation_with=plot_base.fill_null_standard_deviation_with,
        smoothing_mode=plot_base.smoothing_mode,
        smoothing_window_size=plot_base.smoothing_window_size,
        smoothing_min_periods=plot_base.smoothing_min_periods,
        output_path=plot_base.output_path,
    )

In [ ]:
memory_ablation_plot = with_auto_title(
    plotting.NotebookPlot(
        shared_filters={
            "level": 21,
            "rand_prob": 0.3,
            "mercury_split_sensory_raw_latent_valid": True,
        },
        series=[
            plotting.NotebookSeries(
                name="memory_08",
                metric_column_name="latent_precision",
                label="Latent precision",
                filter_values={"weight_memory": 0.8},
                band_alpha=0.20,
            ),
            plotting.NotebookSeries(
                name="memory_10",
                metric_column_name="latent_precision",
                label="Latent precision",
                filter_values={"weight_memory": 1.0},
                line_style="--",
                show_standard_deviation_band=False,
            ),
        ],
        y_axis_label="Precision",
        y_limits=(0.0, 1.0),
        use_grid=True,
    ),
    default_source_series_name="memory_08",
)

plotting.display_notebook_plot(results, memory_ablation_plot)

In [ ]:
validity_ablation_plot = with_auto_title(
    plotting.NotebookPlot(
        shared_filters={
            "level": 21,
            "rand_prob": 0.3,
            "weight_memory": 1.0,
        },
        series=[
            plotting.NotebookSeries(
                name="valid_only_yes",
                metric_column_name="latent_mean_total_variation",
                label="Latent transition precision",
                filter_values={"mercury_split_sensory_raw_latent_valid": True},
                band_alpha=0.25,
            ),
            plotting.NotebookSeries(
                name="valid_only_no",
                metric_column_name="latent_mean_total_variation",
                label="Latent transition precision",
                filter_values={"mercury_split_sensory_raw_latent_valid": False},
                line_style=":",
                show_standard_deviation_band=False,
            ),
        ],
        y_axis_label="Mean total variation",
        use_grid=True,
    ),
    default_source_series_name="valid_only_yes",
)

plotting.display_notebook_plot(results, validity_ablation_plot)

In [ ]:
level_comparison_plot = with_auto_title(
    plotting.NotebookPlot(
        shared_filters={
            "rand_prob": 0.3,
            "weight_memory": 1.0,
            "mercury_split_sensory_raw_latent_valid": True,
        },
        series=[
            plotting.NotebookSeries(
                name="level_13_edge_F1",
                metric_column_name="latent_edge_f1",
                label="Latent edge F1",
                filter_values={"level": 13},
                band_alpha=0.20,
                show_standard_deviation_band=True,
            ),
            plotting.NotebookSeries(
                name="level_13_action_edge_F1",
                metric_column_name="latent_action_conditioned_edge_f1",
                label="Latent action-conditioned edge F1",
                filter_values={"level": 13},
                band_alpha=0.20,
                show_standard_deviation_band=True,
            ),
            plotting.NotebookSeries(
                name="level_13_edge_F1_sensory",
                metric_column_name="sensory_edge_f1",
                label="Sensory action-conditioned edge F1",
                filter_values={"level": 13},
                band_alpha=0.20,
                show_standard_deviation_band=True,
            ),
            plotting.NotebookSeries(
                name="level_13_action_edge_F1_sensory",
                metric_column_name="sensory_action_conditioned_edge_f1",
                label="Sensory action-conditioned edge F1",
                filter_values={"level": 13},
                band_alpha=0.20,
                show_standard_deviation_band=True,
            ),

        ],
        y_axis_label="Edge F1",
        y_limits=(0.0, 1.0),
        use_grid=True,
    ),
    default_source_series_name="level_13",
)

plotting.display_notebook_plot(results, level_comparison_plot)

In [ ]:
levels = [13,20,21]
weights_mem = [0.8, 0.9, 0.95, 0.99, 1.0]
rand_probs = [0.0, 0.3]
valid_onlys = [True, False]
for level in levels:
    for weight_mem in weights_mem:
        for rand_prob in rand_probs:
            for valid_only in valid_onlys:
                level_comparison_plot = plotting.NotebookPlot(
                        shared_filters={
                            "rand_prob": rand_prob,
                            "weight_memory": weight_mem,
                            "mercury_split_sensory_raw_latent_valid": valid_only,
                            "level": level,
                        },
                        title_spec=plotting.NotebookTitleSpec(
                            metric_label_mode="custom",
                            metric_label_text="$TV_{mean}$",
                            separator=" | ",
                            parameter_specs=[
                                plotting.build_title_parameter_spec(
                                    plotting.CANONICAL_TITLE_PARAMETERS[parameter_name]
                                )
                                for parameter_name in plotting.CANONICAL_TITLE_PARAMETER_ORDER
                            ],
                        ),
                        series=[
                           plotting.NotebookSeries(
                                name="sensory_mean_total_variation",
                                metric_column_name="sensory_mean_total_variation",
                                label="Sensory mean total variation",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                            plotting.NotebookSeries(
                                name="latent mean_total_variation",
                                metric_column_name="latent_mean_total_variation",
                                label="Latent mean total variation",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                           plotting.NotebookSeries(
                                name="sensory_action_conditioned_mean_total_variation",
                                metric_column_name="sensory_action_conditioned_mean_total_variation",
                                label="Sensory action-conditioned mean total variation",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                            plotting.NotebookSeries(
                                name="latent_action_conditioned_mean_total_variation",
                                metric_column_name="latent_action_conditioned_mean_total_variation",
                                label="Latent action-conditioned mean total variation",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),



                        ],
                        y_axis_label="Mean total variation",
                        y_limits=(0.0, 1.0),
                        use_grid=False,
                    )
                print(plotting.generate_notebook_plot_title(level_comparison_plot))

                plotting.display_notebook_plot(results, level_comparison_plot)

In [ ]:
levels = [13,20,21]
weights_mem = [0.8, 0.9, 0.95, 0.99, 1.0]
rand_probs = [0.0, 0.3]
valid_onlys = [True, False]
for level in levels:
    for weight_mem in weights_mem:
        for rand_prob in rand_probs:
            for valid_only in valid_onlys:
                level_comparison_plot = plotting.NotebookPlot(
                        shared_filters={
                            "rand_prob": rand_prob,
                            "weight_memory": weight_mem,
                            "mercury_split_sensory_raw_latent_valid": valid_only,
                            "level": level,
                        },
                        title_spec=plotting.NotebookTitleSpec(
                            metric_label_mode="custom",
                            metric_label_text="Edge F1",
                            separator=" | ",
                            parameter_specs=[
                                plotting.build_title_parameter_spec(
                                    plotting.CANONICAL_TITLE_PARAMETERS[parameter_name]
                                )
                                for parameter_name in plotting.CANONICAL_TITLE_PARAMETER_ORDER
                            ],
                        ),
                        series=[
                           plotting.NotebookSeries(
                                name="sensory_edge_F1",
                                metric_column_name="sensory_edge_f1",
                                label="Sensory Edge F1",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                            plotting.NotebookSeries(
                                name="latent mean_total_variation",
                                metric_column_name="latent_edge_f1",
                                label="Latent Edge F1",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                           plotting.NotebookSeries(
                                name="sensory_action_conditioned_edge_F1",
                                metric_column_name="sensory_action_conditioned_edge_f1",
                                label="Sensory action-conditioned Edge F1",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                            plotting.NotebookSeries(
                                name="latent_action_conditioned_edge_F1",
                                metric_column_name="latent_action_conditioned_edge_f1",
                                label="Latent action-conditioned Edge F1",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),



                        ],
                        y_axis_label="Edge F1",
                        y_limits=(0.0, 1.0),
                        use_grid=False,
                    )
                print(plotting.generate_notebook_plot_title(level_comparison_plot))

                plotting.display_notebook_plot(results, level_comparison_plot)

In [ ]:
levels = [13,20,21]
weights_mem = [0.8, 0.9, 0.95, 0.99, 1.0]
rand_probs = [0.0, 0.3]
valid_onlys = [True, False]
for level in levels:
    for weight_mem in weights_mem:
        for rand_prob in rand_probs:
            for valid_only in valid_onlys:
                level_comparison_plot = plotting.NotebookPlot(
                        shared_filters={
                            "rand_prob": rand_prob,
                            "weight_memory": weight_mem,
                            "mercury_split_sensory_raw_latent_valid": valid_only,
                            "level": level,
                        },
                        title_spec=plotting.NotebookTitleSpec(
                            metric_label_mode="custom",
                            metric_label_text="State Representation Precision",
                            separator=" | ",
                            parameter_specs=[
                                plotting.build_title_parameter_spec(
                                    plotting.CANONICAL_TITLE_PARAMETERS[parameter_name]
                                )
                                for parameter_name in plotting.CANONICAL_TITLE_PARAMETER_ORDER
                            ],
                        ),
                        series=[
                           plotting.NotebookSeries(
                                name="sensory_edge_F1",
                                metric_column_name="sensory_precision",
                                label="Sensory State Precision",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),
                            plotting.NotebookSeries(
                                name="latent mean_total_variation",
                                metric_column_name="latent_precision",
                                label="Latent State Precision",
                                band_alpha=0.20,
                                show_standard_deviation_band=True,
                            ),



                        ],
                        y_axis_label="State Representation Precision",
                        y_limits=(0.0, 1.0),
                        use_grid=False,
                    )
                print(plotting.generate_notebook_plot_title(level_comparison_plot))

                plotting.display_notebook_plot(results, level_comparison_plot)

In [ ]:
alignment_level = 20
alignment_rand_prob = 0.3
alignment_weight_memory = 1.0
alignment_valid_only = True

alignment_candidates = dataset.runs.filter(
    (pl.col("level") == alignment_level)
    & (pl.col("rand_prob") == alignment_rand_prob)
    & (pl.col("weight_memory") == alignment_weight_memory)
    & (pl.col("mercury_split_sensory_raw_latent_valid") == alignment_valid_only)
).select([
    "run_id",
    "level",
    "rand_prob",
    "weight_memory",
    "mercury_split_sensory_raw_latent_valid",
    "mercury_paper_precision_json",
    "mercury_states_parquet",
    "mercury_latent_graph_npz",
]).sort("run_id")

if alignment_candidates.is_empty():
    raise ValueError(
        f"No runs found for level={alignment_level}, rand_prob={alignment_rand_prob}, "
        f"weight_memory={alignment_weight_memory}, valid_only={alignment_valid_only}"
    )

alignment_run = alignment_candidates.row(0, named=True)
print(alignment_candidates.head(5))

alignment_demo = plotting.display_alignment_contingency_heatmap_from_paper_precision_json(
    paper_precision_json_path=alignment_run["mercury_paper_precision_json"],
    matrix_key="sensory_exact_state_cooccurrence_matrix",
    title=(
        f"Latent Exact Alignment | run {alignment_run['run_id']} | "
        f"level {alignment_run['level']} | p={alignment_run['rand_prob']} | "
        f"w_mem={alignment_run['weight_memory']} | valid={alignment_run['mercury_split_sensory_raw_latent_valid']}"
    ),
)
alignment_demo = plotting.display_alignment_contingency_heatmap_from_paper_precision_json(
    paper_precision_json_path=alignment_run["mercury_paper_precision_json"],
    matrix_key="latent_exact_state_cooccurrence_matrix",
    title=(
        f"Latent Exact Alignment | run {alignment_run['run_id']} | "
        f"level {alignment_run['level']} | p={alignment_run['rand_prob']} | "
        f"w_mem={alignment_run['weight_memory']} | valid={alignment_run['mercury_split_sensory_raw_latent_valid']}"
    ),
)

print(alignment_run)
print(alignment_demo.contingency)
print(alignment_demo.alignment)


In [ ]:
alignment_history_demo = plotting.display_alignment_history_from_paper_precision_json(
    paper_precision_json_path=alignment_run["mercury_paper_precision_json"],
    states_parquet_path=alignment_run["mercury_states_parquet"],
    matrix_key="latent_exact_state_cooccurrence_matrix",
    title=(
        f"Latent Exact Masked Alignment Over Time (compressed GT axis) | run {alignment_run['run_id']} | "
        f"level {alignment_run['level']}"
    ),
)

print('steps', alignment_history_demo.steps.tolist())
print('assignments_shape', alignment_history_demo.assignments.shape)
print('confidence_shape', alignment_history_demo.confidence.shape)
print('active_mask_shape', alignment_history_demo.active_mask.shape)


In [ ]:
import matplotlib.pyplot as plt
import polars as pl
from pathlib import Path
print(Path.cwd())

levels = [13, 20, 21]
memory_weights = [0.8, 0.9, 0.95, 0.99, 1.0]
random_action_probabilities = [0.0, 0.3]
valid_only_options = [True, False]

condition_label_map = {
    (0.0, False): "rand=0.0 | valid=False",
    (0.0, True): "rand=0.0 | valid=True",
    (0.3, False): "rand=0.3 | valid=False",
    (0.3, True): "rand=0.3 | valid=True",
}


def build_joined_step_metrics_dataframe(
    study_dataset: pl.DataFrame,
    step_metrics: pl.DataFrame,
) -> pl.DataFrame:
    """
    Join config metadata onto per-step metrics through run_id.
    """
    metadata_columns = [
        "run_id",
        "level",
        "rand_prob",
        "weight_memory",
        "mercury_split_sensory_raw_latent_valid",
        "seed",
    ]

    metadata = study_dataset.select(metadata_columns).unique()

    joined = step_metrics.join(
        metadata,
        on="run_id",
        how="inner",
    )

    return joined


def aggregate_metric_over_seeds(
    joined_step_metrics: pl.DataFrame,
    metric_column_name: str,
) -> pl.DataFrame:
    """
    Compute mean and std across seeds for each condition at each step.
    """
    aggregated = (
        joined_step_metrics
        .group_by(
            [
                "level",
                "weight_memory",
                "rand_prob",
                "mercury_split_sensory_raw_latent_valid",
                "step",
            ]
        )
        .agg(
            [
                pl.col(metric_column_name).mean().alias("metric_mean"),
                pl.col(metric_column_name).std().fill_null(0.0).alias("metric_std"),
            ]
        )
        .sort(
            [
                "level",
                "weight_memory",
                "rand_prob",
                "mercury_split_sensory_raw_latent_valid",
                "step",
            ]
        )
    )

    return aggregated


def plot_metric_grid(
    aggregated_sensory_metric: pl.DataFrame,
    aggregated_latent_metric: pl.DataFrame,
    figure_title: str,
    y_axis_label: str,
    y_limits: tuple[float, float] | None = None,
    show_standard_deviation_band: bool = False,
):
    figure, axes = plt.subplots(
        nrows=len(memory_weights),
        ncols=len(levels),
        figsize=(18, 18),
        sharex=True,
        sharey=True,
        constrained_layout=False,
        dpi=200,
    )

    for row_index, memory_weight in enumerate(memory_weights):
        for column_index, level in enumerate(levels):
            axis = axes[row_index, column_index]

            for random_action_probability in random_action_probabilities:
                for valid_only in valid_only_options:
                    sensory_subset = (
                        aggregated_sensory_metric
                        .filter(
                            (pl.col("level") == level)
                            & (pl.col("weight_memory") == memory_weight)
                            & (pl.col("rand_prob") == random_action_probability)
                            & (
                                pl.col("mercury_split_sensory_raw_latent_valid")
                                == valid_only
                            )
                        )
                        .sort("step")
                    )

                    latent_subset = (
                        aggregated_latent_metric
                        .filter(
                            (pl.col("level") == level)
                            & (pl.col("weight_memory") == memory_weight)
                            & (pl.col("rand_prob") == random_action_probability)
                            & (
                                pl.col("mercury_split_sensory_raw_latent_valid")
                                == valid_only
                            )
                        )
                        .sort("step")
                    )

                    if sensory_subset.height == 0 or latent_subset.height == 0:
                        continue

                    condition_label = condition_label_map[
                        (random_action_probability, valid_only)
                    ]

                    sensory_steps = sensory_subset["step"].to_list()
                    sensory_means = sensory_subset["metric_mean"].to_list()
                    sensory_stds = sensory_subset["metric_std"].to_list()

                    latent_steps = latent_subset["step"].to_list()
                    latent_means = latent_subset["metric_mean"].to_list()
                    latent_stds = latent_subset["metric_std"].to_list()

                    sensory_line = axis.plot(
                        sensory_steps,
                        sensory_means,
                        linestyle="-",
                        label=f"Sensory | {condition_label}",
                    )[0]

                    latent_line = axis.plot(
                        latent_steps,
                        latent_means,
                        linestyle="--",
                        label=f"Latent | {condition_label}",
                    )[0]

                    if show_standard_deviation_band:
                        axis.fill_between(
                            sensory_steps,
                            [mean - std for mean, std in zip(sensory_means, sensory_stds)],
                            [mean + std for mean, std in zip(sensory_means, sensory_stds)],
                            alpha=0.10,
                        )
                        axis.fill_between(
                            latent_steps,
                            [mean - std for mean, std in zip(latent_means, latent_stds)],
                            [mean + std for mean, std in zip(latent_means, latent_stds)],
                            alpha=0.10,
                        )

            axis.set_title(f"level={level} | $w_{{mem}}$={memory_weight}", fontsize=16)
            axis.set_xlabel("Timesteps", fontsize=16)
            axis.set_ylabel(y_axis_label, fontsize=16)

            if y_limits is not None:
                axis.set_ylim(*y_limits)

            axis.grid(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        fontsize=18,
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, 1.12),
    )
    figure.suptitle(figure_title, y=1.15, fontsize=20)



    output_directory = Path(Path.cwd() / "results") / "figures"
    output_directory.mkdir(parents=True, exist_ok=True)

    output_filename = figure_title.replace(" ", "_")
    output_path = output_directory / f"{output_filename}.png"

    figure.savefig(output_path, bbox_inches="tight")
    print(f"Saved figure to {output_path}")
    plt.show()


joined_step_metrics = build_joined_step_metrics_dataframe(
    study_dataset=dataset.runs,
    step_metrics=history_with_state_recall,
)

def plot_metric_pair(
    sensory_metric_column_name: str,
    latent_metric_column_name: str,
    figure_title: str,
    y_axis_label: str,
    y_limits: tuple[float, float] | None = (0.0, 1.0),
) -> None:
    aggregated_sensory_metric = aggregate_metric_over_seeds(
        joined_step_metrics=joined_step_metrics,
        metric_column_name=sensory_metric_column_name,
    )
    aggregated_latent_metric = aggregate_metric_over_seeds(
        joined_step_metrics=joined_step_metrics,
        metric_column_name=latent_metric_column_name,
    )
    plot_metric_grid(
        aggregated_sensory_metric=aggregated_sensory_metric,
        aggregated_latent_metric=aggregated_latent_metric,
        figure_title=figure_title,
        y_axis_label=y_axis_label,
        y_limits=y_limits,
        show_standard_deviation_band=True,
    )


plot_metric_pair(
    sensory_metric_column_name="sensory_mean_total_variation",
    latent_metric_column_name="latent_mean_total_variation",
    figure_title="Sensory vs Latent $TV_{mean}$",
    y_axis_label=r"$TV_{mean}$",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_action_conditioned_mean_total_variation",
    latent_metric_column_name="latent_action_conditioned_mean_total_variation",
    figure_title="Sensory vs Latent $TV^{act}_{mean}$",
    y_axis_label="$TV^{act}_{mean}$",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_precision",
    latent_metric_column_name="latent_precision",
    figure_title="Sensory vs Latent State Precision",
    y_axis_label="Precision",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_state_recall",
    latent_metric_column_name="latent_state_recall",
    figure_title="Sensory vs Latent State Recall",
    y_axis_label="State Recall",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_edge_f1",
    latent_metric_column_name="latent_edge_f1",
    figure_title="Sensory vs Latent Edge F1",
    y_axis_label="Edge F1",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_edge_precision",
    latent_metric_column_name="latent_edge_precision",
    figure_title="Sensory vs Latent Edge Precision",
    y_axis_label="Edge Precision",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_edge_recall",
    latent_metric_column_name="latent_edge_recall",
    figure_title="Sensory vs Latent Edge Recall",
    y_axis_label="Edge Recall",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_action_conditioned_edge_f1",
    latent_metric_column_name="latent_action_conditioned_edge_f1",
    figure_title="Sensory vs Latent Action-Conditioned Edge F1",
    y_axis_label="Action-conditioned Edge F1",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_action_conditioned_edge_precision",
    latent_metric_column_name="latent_action_conditioned_edge_precision",
    figure_title="Sensory vs Latent Action-Conditioned Edge Precision",
    y_axis_label="Action-conditioned Edge Precision",
)

plot_metric_pair(
    sensory_metric_column_name="sensory_action_conditioned_edge_recall",
    latent_metric_column_name="latent_action_conditioned_edge_recall",
    figure_title="Sensory vs Latent Action-Conditioned Edge Recall",
    y_axis_label="Action-conditioned Edge Recall",
)

In [ ]:
# Final metric reporting: absolute context plus regime-organised deltas.
# TMGWR is the sensory baseline; MERCURY is the latent representation.
# TMGWR is policy-only here, so it is shown once per level/rand while MERCURY is split by valid-only setting and memory weight.

from pathlib import Path
from IPython.display import Markdown, display

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

METHOD_LABELS = {
    "tmgwr": "TMGWR",
    "mercury": "MERCURY",
}

FIGURE_METRICS = [
    {"label": "State precision", "slug": "state_precision", "tmgwr_column": "sensory_precision", "mercury_column": "latent_precision", "lower_is_better": False},
    {"label": "State recall", "slug": "state_recall", "tmgwr_column": "sensory_state_recall", "mercury_column": "latent_state_recall", "lower_is_better": False},
    {"label": "Mean total variation", "slug": "mean_total_variation", "tmgwr_column": "sensory_mean_total_variation", "mercury_column": "latent_mean_total_variation", "lower_is_better": True},
    {"label": "Action-cond. mean TV", "slug": "action_cond_mean_total_variation", "tmgwr_column": "sensory_action_conditioned_mean_total_variation", "mercury_column": "latent_action_conditioned_mean_total_variation", "lower_is_better": True},
    {"label": "Edge precision", "slug": "edge_precision", "tmgwr_column": "sensory_edge_precision", "mercury_column": "latent_edge_precision", "lower_is_better": False},
    {"label": "Edge recall", "slug": "edge_recall", "tmgwr_column": "sensory_edge_recall", "mercury_column": "latent_edge_recall", "lower_is_better": False},
    {"label": "Edge F1", "slug": "edge_f1", "tmgwr_column": "sensory_edge_f1", "mercury_column": "latent_edge_f1", "lower_is_better": False},
    {"label": "Action-cond. edge precision", "slug": "action_cond_edge_precision", "tmgwr_column": "sensory_action_conditioned_edge_precision", "mercury_column": "latent_action_conditioned_edge_precision", "lower_is_better": False},
    {"label": "Action-cond. edge recall", "slug": "action_cond_edge_recall", "tmgwr_column": "sensory_action_conditioned_edge_recall", "mercury_column": "latent_action_conditioned_edge_recall", "lower_is_better": False},
    {"label": "Action-cond. edge F1", "slug": "action_cond_edge_f1", "tmgwr_column": "sensory_action_conditioned_edge_f1", "mercury_column": "latent_action_conditioned_edge_f1", "lower_is_better": False},
]

METADATA_COLUMNS = [
    "level",
    "weight_memory",
    "rand_prob",
    "mercury_split_sensory_raw_latent_valid",
]
PANEL_REGIME_COLUMNS = ["rand_prob"]
BASE_COLUMNS = ["level", *PANEL_REGIME_COLUMNS]
MEMORY_WEIGHTS = [0.8, 0.9, 0.95, 0.99, 1.0]
MEMORY_WEIGHT_KEYS = {0.8: "08", 0.9: "09", 0.95: "095", 0.99: "099", 1.0: "10"}
MEMORY_WEIGHT_LABELS = {0.8: "0.8", 0.9: "0.9", 0.95: "0.95", 0.99: "0.99", 1.0: "1.0"}
MERCURY_COMBINATIONS = [
    {"valid": valid, "weight": weight, "key": f"{valid_key}_w{weight_key}", "label": f"{valid_label} w={MEMORY_WEIGHT_LABELS[weight]}"}
    for valid, valid_key, valid_label in [(False, "raw", "raw"), (True, "valid", "valid")]
    for weight in MEMORY_WEIGHTS
    for weight_key in [MEMORY_WEIGHT_KEYS[weight]]
]

TMGWR_COLOR = "#4E79A7"
MERCURY_COLORS = {
    "raw_w08": "#FDB462",
    "raw_w09": "#F6A057",
    "raw_w095": "#F28E2B",
    "raw_w099": "#D9741F",
    "raw_w10": "#B85C1A",
    "valid_w08": "#C2A5CF",
    "valid_w09": "#A98BC0",
    "valid_w095": "#8F70AF",
    "valid_w099": "#7B3294",
    "valid_w10": "#5E1F75",
}
GOOD_BACKGROUND_COLOR = "#2E7D32"
BAD_BACKGROUND_COLOR = "#C62828"
ZERO_COLOR = "#4D4D4D"
EDGE_COLOR = "#262626"
OUTPUT_DIRECTORY = REPO_ROOT / "results" / "current" / "notebook_plots"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


def _final_step_per_run(history_frame: pl.DataFrame) -> pl.DataFrame:
    final_steps = history_frame.group_by("run_id").agg(pl.col("step").max().alias("step"))
    return history_frame.join(final_steps, on=["run_id", "step"], how="inner")


def _level_label(level: int | float) -> str:
    return f"L{int(level)}"


def _panel_label(row: dict[str, object]) -> str:
    return f"p={float(row['rand_prob']):.1f}"


def _format_stat(mean: float | None, sd: float | None) -> str:
    if mean is None:
        return "NA"
    if sd is None:
        return f"{mean:.3f} ± NA"
    return f"{mean:.3f} ± {sd:.3f}"


def _check_metric_columns(metric: dict[str, object]) -> None:
    missing = [
        column
        for column in [str(metric["tmgwr_column"]), str(metric["mercury_column"]), *METADATA_COLUMNS, "run_id", "seed"]
        if column not in final_metrics_with_metadata.columns
    ]
    if missing:
        raise ValueError(f"Missing columns for {metric['label']}: {missing}")


def _one_mercury_summary(frame: pl.DataFrame, *, combination: dict[str, object], value_column: str, prefix: str) -> pl.DataFrame:
    key = str(combination["key"])
    return (
        frame
        .filter(
            (pl.col("weight_memory") == float(combination["weight"]))
            & (pl.col("mercury_split_sensory_raw_latent_valid") == bool(combination["valid"]))
        )
        .group_by(BASE_COLUMNS)
        .agg([
            pl.len().alias(f"{prefix}_{key}_n"),
            pl.col(value_column).mean().alias(f"{prefix}_{key}_mean"),
            pl.col(value_column).std().fill_null(0.0).alias(f"{prefix}_{key}_sd"),
        ])
    )


def build_absolute_metric_frame(metric: dict[str, object]) -> pl.DataFrame:
    _check_metric_columns(metric)
    tmgwr_column = str(metric["tmgwr_column"])
    mercury_column = str(metric["mercury_column"])

    tmgwr_by_seed = (
        final_metrics_with_metadata
        .select(["run_id", "seed", *BASE_COLUMNS, tmgwr_column])
        .drop_nulls(tmgwr_column)
        .group_by([*BASE_COLUMNS, "seed"])
        .agg(pl.col(tmgwr_column).mean().alias("tmgwr_value"))
    )
    tmgwr_summary = (
        tmgwr_by_seed
        .group_by(BASE_COLUMNS)
        .agg([
            pl.len().alias("tmgwr_n"),
            pl.col("tmgwr_value").mean().alias("tmgwr_mean"),
            pl.col("tmgwr_value").std().fill_null(0.0).alias("tmgwr_sd"),
        ])
    )

    mercury_frame = (
        final_metrics_with_metadata
        .select(["run_id", "seed", *BASE_COLUMNS, "weight_memory", "mercury_split_sensory_raw_latent_valid", mercury_column])
        .drop_nulls(mercury_column)
        .rename({mercury_column: "mercury_value"})
    )

    combined = tmgwr_summary
    for combination in MERCURY_COMBINATIONS:
        combined = combined.join(
            _one_mercury_summary(mercury_frame, combination=combination, value_column="mercury_value", prefix="mercury"),
            on=BASE_COLUMNS,
            how="left",
        )

    return combined.sort(BASE_COLUMNS)


def build_delta_metric_frame(metric: dict[str, object]) -> pl.DataFrame:
    _check_metric_columns(metric)
    tmgwr_column = str(metric["tmgwr_column"])
    mercury_column = str(metric["mercury_column"])

    delta_frame = (
        final_metrics_with_metadata
        .select([
            "run_id",
            "seed",
            *BASE_COLUMNS,
            "weight_memory",
            "mercury_split_sensory_raw_latent_valid",
            ((pl.col(mercury_column) - pl.col(tmgwr_column)) * 100.0).alias("delta_pct_points"),
        ])
        .drop_nulls("delta_pct_points")
    )

    combined = delta_frame.select(BASE_COLUMNS).unique()
    for combination in MERCURY_COMBINATIONS:
        combined = combined.join(
            _one_mercury_summary(delta_frame, combination=combination, value_column="delta_pct_points", prefix="delta"),
            on=BASE_COLUMNS,
            how="left",
        )

    return combined.sort(BASE_COLUMNS)


def _series_from_rows(rows: list[dict[str, object]], column_name: str) -> np.ndarray:
    return np.array([float(row[column_name] or 0.0) for row in rows])


def _label_vertical_bars(axis, bars, means: np.ndarray, sds: np.ndarray) -> None:
    for bar, mean, sd in zip(bars, means, sds):
        label_y = min(1.055, float(mean) + float(sd) + 0.018)
        axis.text(
            bar.get_x() + bar.get_width() / 2.0,
            label_y,
            f"{mean:.2f}",
            ha="center",
            va="bottom",
            fontsize=7.0,
            color="black",
        )


def plot_absolute_metric_bars(metric: dict[str, object], absolute_frame: pl.DataFrame) -> Path:
    panels = absolute_frame.select(PANEL_REGIME_COLUMNS).unique().sort(PANEL_REGIME_COLUMNS).to_dicts()
    ncols = len(panels)
    figure, axes = plt.subplots(
        nrows=1,
        ncols=ncols,
        figsize=(14.5, 7.0),
        sharey=True,
        constrained_layout=False,
        dpi=200,
    )
    axes = np.asarray(axes).reshape(-1)

    width = 0.065
    offsets = (np.arange(len(MERCURY_COMBINATIONS) + 1, dtype=float) - len(MERCURY_COMBINATIONS) / 2.0) * width
    for axis, panel in zip(axes, panels):
        panel_frame = absolute_frame.filter(pl.col("rand_prob") == panel["rand_prob"]).sort("level")
        rows = panel_frame.to_dicts()
        x_positions = np.arange(len(rows), dtype=float)

        tmgwr_means = _series_from_rows(rows, "tmgwr_mean")
        tmgwr_sds = _series_from_rows(rows, "tmgwr_sd")
        tmgwr_bars = axis.bar(
            x_positions + offsets[0],
            tmgwr_means,
            width=width,
            yerr=tmgwr_sds,
            capsize=3,
            color=TMGWR_COLOR,
            edgecolor=EDGE_COLOR,
            linewidth=0.6,
            label=METHOD_LABELS["tmgwr"],
        )
        _label_vertical_bars(axis, tmgwr_bars, tmgwr_means, tmgwr_sds)

        for index, combination in enumerate(MERCURY_COMBINATIONS, start=1):
            key = str(combination["key"])
            mercury_means = _series_from_rows(rows, f"mercury_{key}_mean")
            mercury_sds = _series_from_rows(rows, f"mercury_{key}_sd")
            mercury_bars = axis.bar(
                x_positions + offsets[index],
                mercury_means,
                width=width,
                yerr=mercury_sds,
                capsize=3,
                color=MERCURY_COLORS[key],
                edgecolor=EDGE_COLOR,
                linewidth=0.6,
                label=f"{METHOD_LABELS['mercury']} {combination['label']}",
            )
            _label_vertical_bars(axis, mercury_bars, mercury_means, mercury_sds)

        axis.set_title(_panel_label(panel), fontsize=10)
        axis.set_xticks(x_positions, [_level_label(row["level"]) for row in rows])
        axis.set_ylim(0.0, 1.08)
        axis.grid(axis="y", alpha=0.18)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.94), fontsize=8)
    figure.supylabel(metric["label"], fontsize=10)
    figure.suptitle(f"{metric['label']} absolute performance by level", fontsize=13, y=0.985)
    figure.tight_layout(rect=(0.02, 0.02, 1.0, 0.84))

    output_path = OUTPUT_DIRECTORY / f"absolute_{metric['slug']}_by_level_faceted_regime.png"
    figure.savefig(output_path, bbox_inches="tight", facecolor="white", edgecolor="white")
    print(f"Saved absolute plot: {output_path}")
    plt.show()
    return output_path


def plot_delta_metric_bars(metric: dict[str, object], delta_frame: pl.DataFrame) -> Path:
    panels = delta_frame.select(PANEL_REGIME_COLUMNS).unique().sort(PANEL_REGIME_COLUMNS).to_dicts()
    ncols = len(panels)
    figure, axes = plt.subplots(
        nrows=1,
        ncols=ncols,
        figsize=(14.5, 6.4),
        sharex=True,
        constrained_layout=False,
        dpi=200,
    )
    axes = np.asarray(axes).reshape(-1)

    mean_columns = [f"delta_{combination['key']}_mean" for combination in MERCURY_COMBINATIONS]
    sd_columns = [f"delta_{combination['key']}_sd" for combination in MERCURY_COMBINATIONS]
    x_min = min(float((delta_frame[mean] - delta_frame[sd]).min()) for mean, sd in zip(mean_columns, sd_columns))
    x_max = max(float((delta_frame[mean] + delta_frame[sd]).max()) for mean, sd in zip(mean_columns, sd_columns))
    x_padding = max(2.0, (x_max - x_min) * 0.18)
    x_limits = (min(0.0, x_min) - x_padding, max(0.0, x_max) + x_padding)
    label_padding = max(0.8, (x_limits[1] - x_limits[0]) * 0.012)

    bar_height = 0.075
    y_offsets = (np.arange(len(MERCURY_COMBINATIONS), dtype=float) - (len(MERCURY_COMBINATIONS) - 1) / 2.0) * bar_height
    for axis, panel in zip(axes, panels):
        panel_frame = delta_frame.filter(pl.col("rand_prob") == panel["rand_prob"]).sort("level")
        rows = panel_frame.to_dicts()
        y_positions = np.arange(len(rows), dtype=float)

        for offset, combination in zip(y_offsets, MERCURY_COMBINATIONS):
            key = str(combination["key"])
            means = _series_from_rows(rows, f"delta_{key}_mean")
            sds = _series_from_rows(rows, f"delta_{key}_sd")
            bars = axis.barh(
                y_positions + offset,
                means,
                height=bar_height,
                xerr=sds,
                capsize=3,
                color=MERCURY_COLORS[key],
                edgecolor=EDGE_COLOR,
                linewidth=0.6,
                label=str(combination["label"]),
            )
            for bar, mean, sd in zip(bars, means, sds):
                label_x = mean + sd + label_padding if mean >= 0.0 else mean - sd - label_padding
                ha = "left" if mean >= 0.0 else "right"
                axis.text(label_x, bar.get_y() + bar.get_height() / 2.0, f"{mean:+.1f}", ha=ha, va="center", fontsize=8, color="black")

        if metric["lower_is_better"]:
            axis.axvspan(x_limits[0], 0.0, color=GOOD_BACKGROUND_COLOR, alpha=0.16, zorder=0)
            axis.axvspan(0.0, x_limits[1], color=BAD_BACKGROUND_COLOR, alpha=0.16, zorder=0)
        else:
            axis.axvspan(x_limits[0], 0.0, color=BAD_BACKGROUND_COLOR, alpha=0.16, zorder=0)
            axis.axvspan(0.0, x_limits[1], color=GOOD_BACKGROUND_COLOR, alpha=0.16, zorder=0)
        axis.axvline(0.0, color=ZERO_COLOR, linewidth=1.0)
        axis.set_title(_panel_label(panel), fontsize=10)
        axis.set_yticks(y_positions, [_level_label(row["level"]) for row in rows])
        axis.invert_yaxis()
        axis.set_xlim(*x_limits)
        axis.grid(axis="x", alpha=0.18)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    delta_handles = [
        mpatches.Patch(facecolor=MERCURY_COLORS[str(combination["key"])], edgecolor=EDGE_COLOR, label=str(combination["label"]))
        for combination in MERCURY_COMBINATIONS
    ]
    figure.legend(handles=delta_handles, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 0.94), fontsize=8)
    figure.supxlabel(f"{METHOD_LABELS['mercury']} - {METHOD_LABELS['tmgwr']} (percentage points)", fontsize=10)
    figure.suptitle(f"{metric['label']} delta by regime", fontsize=13, y=0.985)
    figure.tight_layout(rect=(0.02, 0.02, 1.0, 0.80))

    output_path = OUTPUT_DIRECTORY / f"delta_{metric['slug']}_by_regime.png"
    figure.savefig(output_path, bbox_inches="tight", facecolor="white", edgecolor="white")
    print(f"Saved delta plot: {output_path}")
    plt.show()
    return output_path


def metric_summary_table(metric: dict[str, object], absolute_frame: pl.DataFrame) -> pl.DataFrame:
    expressions = [
        pl.lit(metric["label"]).alias("metric"),
        pl.concat_str([pl.lit("L"), pl.col("level").cast(pl.Utf8)]).alias("level_label"),
        pl.col("rand_prob").map_elements(lambda value: f"{float(value):.1f}", return_dtype=pl.Utf8).alias("rand"),
        pl.struct(["tmgwr_mean", "tmgwr_sd"]).map_elements(lambda row: _format_stat(row["tmgwr_mean"], row["tmgwr_sd"]), return_dtype=pl.Utf8).alias(METHOD_LABELS["tmgwr"]),
    ]
    mercury_columns = []
    for combination in MERCURY_COMBINATIONS:
        key = str(combination["key"])
        label = f"{METHOD_LABELS['mercury']} {combination['label']}"
        mercury_columns.append(label)
        mean_column = f"mercury_{key}_mean"
        sd_column = f"mercury_{key}_sd"
        expressions.append(
            pl.struct([mean_column, sd_column]).map_elements(
                lambda row, mean_column=mean_column, sd_column=sd_column: _format_stat(row[mean_column], row[sd_column]),
                return_dtype=pl.Utf8,
            ).alias(label)
        )

    return absolute_frame.with_columns(expressions).select(["level_label", "rand", "metric", "tmgwr_n", METHOD_LABELS["tmgwr"], *mercury_columns]).rename({"tmgwr_n": "n"})


def to_markdown_table(summary_frame: pl.DataFrame) -> str:
    headers = ["level", "rand", "metric", "n", METHOD_LABELS["tmgwr"], *[f"{METHOD_LABELS['mercury']} {combination['label']}" for combination in MERCURY_COMBINATIONS]]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in summary_frame.iter_rows(named=True):
        lines.append(
            "| "
            + " | ".join([
                str(row["level_label"]),
                str(row["rand"]),
                str(row["metric"]),
                str(row["n"]),
                str(row[METHOD_LABELS["tmgwr"]]),
                *[str(row[f"{METHOD_LABELS['mercury']} {combination['label']}"]) for combination in MERCURY_COMBINATIONS],
            ])
            + " |"
        )
    return "\n".join(lines)


final_metrics = _final_step_per_run(history_with_state_recall)
final_metrics_with_metadata = final_metrics.join(dataset.run_metadata([*METADATA_COLUMNS, "seed"]), on="run_id", how="left", coalesce=True)

absolute_frames = {str(metric["slug"]): build_absolute_metric_frame(metric) for metric in FIGURE_METRICS}
delta_frames = {str(metric["slug"]): build_delta_metric_frame(metric) for metric in FIGURE_METRICS}

absolute_plot_paths = []
delta_plot_paths = []
for metric in FIGURE_METRICS:
    slug = str(metric["slug"])
    absolute_plot_paths.append(plot_absolute_metric_bars(metric, absolute_frames[slug]))
    delta_plot_paths.append(plot_delta_metric_bars(metric, delta_frames[slug]))

precision_metric = next(metric for metric in FIGURE_METRICS if metric["slug"] == "state_precision")
compact_precision_summary = metric_summary_table(precision_metric, absolute_frames[str(precision_metric["slug"])]).sort(["level_label", "rand"])
compact_precision_summary_markdown = to_markdown_table(compact_precision_summary)
summary_path = OUTPUT_DIRECTORY / "compact_precision_summary.md"
summary_path.write_text(compact_precision_summary_markdown + "\n", encoding="utf-8")

display(Markdown(compact_precision_summary_markdown))
print(f"Generated {len(absolute_plot_paths)} absolute plots and {len(delta_plot_paths)} delta plots.")
print(f"Saved compact precision table: {summary_path}")


In [ ]:
# Mercury diagnostics: attribution, lifetimes, and structure events.

from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
import numpy as np
import polars as pl


def _display_saved_png(output_path: Path) -> None:
    try:
        from IPython.display import Image, display
    except ImportError:
        return
    display(Image(filename=str(output_path), embed=True))

STEP_BIN_SIZE = 100
DIAGNOSTIC_OUTPUT_DIRECTORY = Path.cwd() / "results" / "figures"
DIAGNOSTIC_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

DIAGNOSTIC_METADATA_COLUMNS = [
    "level",
    "rand_prob",
    "weight_memory",
    "mercury_split_sensory_raw_latent_valid",
    "seed",
    "run_status",
]
DIAGNOSTIC_GROUP_COLUMNS = [
    "level",
    "rand_prob",
    "weight_memory",
    "mercury_split_sensory_raw_latent_valid",
]
DIAGNOSTIC_MEMORY_WEIGHTS = [0.8, 0.9, 0.95, 0.99, 1.0]
DIAGNOSTIC_LEVELS = [13, 21, 20]
DIAGNOSTIC_LEVEL_LABELS = {13: "3x3", 20: "1x5", 21: "1x4"}
DIAGNOSTIC_RAND_PROBS = [0.0, 0.3]
DIAGNOSTIC_VALID_ONLY_OPTIONS = [True, False]


def _diagnostic_level_label(level: int | float) -> str:
    return DIAGNOSTIC_LEVEL_LABELS.get(int(level), f"L{int(level)}")


def _diagnostic_probability_label(rand_prob: float) -> str:
    return f"p={float(rand_prob):.1f}"


DIAGNOSTIC_WEIGHT_COLORS = {
    0.8: "#4E79A7",
    0.9: "#59A14F",
    0.95: "#F28E2B",
    0.99: "#E15759",
    1.0: "#7B3294",
}

ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN = "selected_memory_drive_raw"
RAW_MEMORY_DRIVE_WEIGHT = 1.0
RAW_MEMORY_DRIVE_VALID_ONLY = False
RAW_MEMORY_DRIVE_PLOT_SPECS = [
    {
        "valid_only": False,
        "title": "Mercury BMU Memory Drive Distribution",
        "output_filename": "mercury_weighted_drives_unbinned.png",
    },
    {
        "valid_only": True,
        "title": "Mercury BMU Memory Drive Distribution | Collision Conditioned",
        "output_filename": "mercury_weighted_drives_unbinned_valid_true.png",
    },
]
RAW_MEMORY_DRIVE_STEP_MIN = 1
RAW_MEMORY_DRIVE_STEP_MAX = 20_000
RAW_MEMORY_DRIVE_STEP_TICKS = [0, 5_000, 10_000, 15_000, 20_000]
RAW_MEMORY_DRIVE_MEDIAN_COLOR = "#2F5F8F"
RAW_MEMORY_DRIVE_QUARTILE_COLOR = "#7FA6C9"
ATTRIBUTION_SHARE_COLUMNS = [
    "selected_memory_drive_share",
    "selected_action_drive_share",
    "selected_undirected_drive_share",
    "selected_baseline_drive_share",
]
ATTRIBUTION_SUPPORT_COLUMNS = [
    "selected_total_support_pre_trace",
    "selected_total_support_post_trace",
    "selected_trace_penalty",
]
AGE_BIN_SIZE = 100
NODE_AGE_DISTRIBUTION_COLUMN = "node_age_distribution"
EDGE_AGE_DISTRIBUTION_COLUMN = "edge_age_distribution"
EVENT_COUNT_COLUMNS = ["clone_created_count", "node_prune_count", "edge_prune_count"]
EVENT_CUMULATIVE_COLUMNS = ["clone_created_cumulative", "node_prune_cumulative", "edge_prune_cumulative"]

DIAGNOSTIC_LABELS = {
    "selected_memory_drive_weighted": "memory weighted",
    "selected_action_drive_weighted": "action weighted",
    "selected_undirected_drive_weighted": "undirected weighted",
    "selected_baseline_drive_weighted": "baseline weighted",
    "selected_memory_drive_share": "memory share",
    "selected_action_drive_share": "action share",
    "selected_undirected_drive_share": "undirected share",
    "selected_baseline_drive_share": "baseline share",
    "selected_total_support_pre_trace": "support pre-trace",
    "selected_total_support_post_trace": "support post-trace",
    "selected_trace_penalty": "trace penalty",
    "node_age_distribution": "node age distribution",
    "edge_age_distribution": "edge age distribution",
    "clone_created_count": "clones created",
    "node_prune_count": "nodes pruned",
    "edge_prune_count": "edges pruned",
    "clone_created_cumulative": "cumulative clones",
    "node_prune_cumulative": "cumulative node prunes",
    "edge_prune_cumulative": "cumulative edge prunes",
    "node_efficiency_ratio": "latent states / GT state",
    "node_age_mean": "mean node age",
    "edge_age_mean": "mean edge age",
}


def _with_step_bin(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.with_columns(
        ((pl.col("step") // STEP_BIN_SIZE) * STEP_BIN_SIZE).cast(pl.Int64).alias("step_bin")
    )


def _aggregate_by_step(
    frame: pl.DataFrame,
    metric_columns: list[str],
    *,
    reducer: str,
) -> pl.DataFrame:
    if reducer == "mean":
        per_run_expressions = [pl.col(column).mean().alias(column) for column in metric_columns]
    elif reducer == "sum":
        per_run_expressions = [pl.col(column).sum().alias(column) for column in metric_columns]
    elif reducer == "max":
        per_run_expressions = [pl.col(column).max().alias(column) for column in metric_columns]
    else:
        raise ValueError(f"Unknown reducer: {reducer}")

    per_run = (
        _with_step_bin(frame)
        .group_by([*DIAGNOSTIC_GROUP_COLUMNS, "seed", "run_id", "step_bin"])
        .agg(per_run_expressions)
    )

    aggregated = (
        per_run
        .group_by([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
        .agg(
            [
                *[pl.col(column).mean().alias(f"{column}_mean") for column in metric_columns],
                *[pl.col(column).std().fill_null(0.0).alias(f"{column}_sd") for column in metric_columns],
                pl.col("seed").n_unique().alias("n_seeds"),
            ]
        )
        .sort([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
    )
    return aggregated


def _node_efficiency_ratio_by_step(history_frame: pl.DataFrame) -> pl.DataFrame:
    ratio_frame = history_frame.with_columns(
        pl.when(pl.col("latent_ground_truth_state_count") > 0)
        .then(pl.col("latent_inferred_state_count") / pl.col("latent_ground_truth_state_count"))
        .otherwise(None)
        .alias("node_efficiency_ratio")
    )
    return _aggregate_by_step(ratio_frame, ["node_efficiency_ratio"], reducer="mean")


def _line_panel_subset(
    frame: pl.DataFrame,
    *,
    level: int,
    weight_memory: float,
    rand_prob: float,
    valid_only: bool,
    metric_column: str,
) -> pl.DataFrame:
    return (
        frame
        .filter(
            (pl.col("level") == level)
            & (pl.col("weight_memory") == weight_memory)
            & (pl.col("rand_prob") == rand_prob)
            & (pl.col("mercury_split_sensory_raw_latent_valid") == valid_only)
        )
        .select(["step_bin", f"{metric_column}_mean", f"{metric_column}_sd"])
        .sort("step_bin")
    )


def plot_metric_lines(
    aggregated_frame: pl.DataFrame,
    metric_columns: list[str],
    *,
    figure_title: str,
    output_filename_prefix: str,
    y_axis_label: str,
    y_limits: tuple[float, float] | None = None,
    show_sd_band: bool = True,
    band_floor: float | None = None,
) -> list[Path]:
    panel_specs = [(rand_prob, valid_only) for rand_prob in DIAGNOSTIC_RAND_PROBS for valid_only in DIAGNOSTIC_VALID_ONLY_OPTIONS]
    metric_colors = plt.get_cmap("tab10")(np.linspace(0.0, 1.0, max(2, len(metric_columns))))
    output_paths: list[Path] = []

    for level in DIAGNOSTIC_LEVELS:
        figure, axes = plt.subplots(
            nrows=len(DIAGNOSTIC_MEMORY_WEIGHTS),
            ncols=len(panel_specs),
            figsize=(8.8, max(3.8, 1.85 * len(DIAGNOSTIC_MEMORY_WEIGHTS) + 1.6)),
            sharex=True,
            sharey=True,
            squeeze=False,
            dpi=150,
        )

        for row_index, weight_memory in enumerate(DIAGNOSTIC_MEMORY_WEIGHTS):
            for column_index, (rand_prob, valid_only) in enumerate(panel_specs):
                axis = axes[row_index, column_index]
                for metric_index, metric_column in enumerate(metric_columns):
                    subset = _line_panel_subset(
                        aggregated_frame,
                        level=level,
                        weight_memory=weight_memory,
                        rand_prob=rand_prob,
                        valid_only=valid_only,
                        metric_column=metric_column,
                    )
                    if subset.is_empty():
                        continue
                    steps = subset.get_column("step_bin").to_numpy()
                    means = subset.get_column(f"{metric_column}_mean").to_numpy()
                    sds = subset.get_column(f"{metric_column}_sd").to_numpy()
                    color = metric_colors[metric_index]
                    axis.plot(steps, means, color=color, linewidth=1.1, label=DIAGNOSTIC_LABELS.get(metric_column, metric_column))
                    if show_sd_band:
                        band_lower = means - sds
                        if band_floor is not None:
                            band_lower = np.maximum(band_lower, band_floor)
                        axis.fill_between(steps, band_lower, means + sds, color=color, alpha=0.10, linewidth=0)
                if row_index == 0:
                    axis.set_title(f"p={rand_prob:g} valid={valid_only}", fontsize=8)
                if column_index == 0:
                    axis.set_ylabel(f"w={weight_memory:g}", fontsize=8)
                if y_limits is not None:
                    axis.set_ylim(*y_limits)
                axis.grid(alpha=0.16)
                axis.tick_params(axis="both", labelsize=7)

        handles, labels = axes[0, 0].get_legend_handles_labels()
        if handles:
            figure.legend(
                handles,
                labels,
                loc="lower center",
                bbox_to_anchor=(0.5, 0.02),
                ncol=min(4, len(metric_columns)),
                frameon=False,
                fontsize=7,
                handlelength=1.8,
                columnspacing=1.0,
            )
        figure.supxlabel(f"training step ({STEP_BIN_SIZE}-step bins)", y=0.08, fontsize=9)
        figure.supylabel(y_axis_label, x=0.01, fontsize=9)
        figure.suptitle(f"{figure_title} | L{level}", y=0.985, fontsize=11)
        figure.tight_layout(rect=(0.04, 0.13, 0.99, 0.94))

        output_path = DIAGNOSTIC_OUTPUT_DIRECTORY / f"{output_filename_prefix}_level{level}.png"
        figure.savefig(output_path, bbox_inches="tight", facecolor="white", edgecolor="white")
        plt.close(figure)
        _display_saved_png(output_path)
        print(f"Saved diagnostic line plot: {output_path}")
        output_paths.append(output_path)

    return output_paths





def raw_memory_drive_source_frame(attribution_frame: pl.DataFrame, *, valid_only: bool) -> pl.DataFrame:
    source = (
        attribution_frame
        .filter(
            (pl.col("weight_memory") == RAW_MEMORY_DRIVE_WEIGHT)
            & (pl.col("mercury_split_sensory_raw_latent_valid") == valid_only)
            & pl.col("level").is_in(DIAGNOSTIC_LEVELS)
            & pl.col("rand_prob").is_in(DIAGNOSTIC_RAND_PROBS)
            & (pl.col("run_status") == "completed")
        )
        .select(["level", "rand_prob", "seed", "run_id", "step", ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN])
    )

    expected_run_count = len(DIAGNOSTIC_LEVELS) * len(DIAGNOSTIC_RAND_PROBS) * 10
    source_run_count = source.select("run_id").unique().height
    if source_run_count != expected_run_count:
        raise AssertionError(f"Expected {expected_run_count} raw-memory-drive runs, found {source_run_count}")

    per_run_steps = (
        source
        .group_by(["level", "rand_prob", "seed", "run_id"])
        .agg([
            pl.col("step").min().alias("min_step"),
            pl.col("step").max().alias("max_step"),
            pl.col("step").n_unique().alias("n_steps"),
        ])
    )
    bad_step_ranges = per_run_steps.filter(
        (pl.col("min_step") != RAW_MEMORY_DRIVE_STEP_MIN)
        | (pl.col("max_step") != RAW_MEMORY_DRIVE_STEP_MAX)
        | (pl.col("n_steps") != RAW_MEMORY_DRIVE_STEP_MAX - RAW_MEMORY_DRIVE_STEP_MIN + 1)
    )
    if not bad_step_ranges.is_empty():
        raise AssertionError("Raw memory drive source does not span exact steps 1..20000 for every run")

    return source


def _raw_memory_drive_fraction_matrix(
    source_frame: pl.DataFrame,
    *,
    level: int,
    rand_prob: float,
    drive_values: list[float],
) -> np.ndarray:
    points = (
        source_frame
        .filter((pl.col("level") == level) & (pl.col("rand_prob") == rand_prob))
        .group_by(["step", ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN])
        .agg((pl.len() / 10.0).alias("seed_fraction"))
        .sort([ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN, "step"])
    )
    if points.is_empty():
        return np.empty((0, 0))

    drive_index = {value: index for index, value in enumerate(drive_values)}
    matrix = np.zeros((len(drive_values), RAW_MEMORY_DRIVE_STEP_MAX - RAW_MEMORY_DRIVE_STEP_MIN + 1), dtype=np.float32)
    for row in points.iter_rows(named=True):
        step_index = int(row["step"]) - RAW_MEMORY_DRIVE_STEP_MIN
        drive_value = float(row[ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN])
        matrix[drive_index[drive_value], step_index] = float(row["seed_fraction"])
    return matrix


def plot_raw_memory_drive_stacked_area(
    source_frame: pl.DataFrame,
    *,
    figure_title: str,
    output_filename: str,
) -> list[Path]:
    plot_frame = (
        source_frame
        .sort(["run_id", "step"])
        .with_columns(
            pl.when(pl.col(ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN) == 0.0)
            .then(None)
            .otherwise(pl.col(ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN))
            .forward_fill()
            .backward_fill()
            .over("run_id")
            .alias(ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN)
        )
    )
    drive_values = sorted(float(value) for value in plot_frame.get_column(ATTRIBUTION_RAW_MEMORY_DRIVE_COLUMN).unique().to_list())
    fraction_ticks = np.linspace(0.0, 1.0, 6)
    color_map = plt.get_cmap("viridis")
    normalizer = Normalize(vmin=0.0, vmax=1.0)
    figure, axes = plt.subplots(
        nrows=len(DIAGNOSTIC_LEVELS),
        ncols=len(DIAGNOSTIC_RAND_PROBS),
        figsize=(7.0, 9.0),
        sharex=True,
        sharey=True,
        squeeze=False,
        dpi=200,
    )

    for row_index, level in enumerate(DIAGNOSTIC_LEVELS):
        for column_index, rand_prob in enumerate(DIAGNOSTIC_RAND_PROBS):
            axis = axes[row_index, column_index]
            matrix = _raw_memory_drive_fraction_matrix(
                plot_frame,
                level=level,
                rand_prob=rand_prob,
                drive_values=drive_values,
            )
            if matrix.size:
                axis.imshow(
                    matrix,
                    cmap=color_map,
                    norm=normalizer,
                    aspect="auto",
                    origin="lower",
                    interpolation="nearest",
                    resample=False,
                    extent=(
                        RAW_MEMORY_DRIVE_STEP_MIN - 0.5,
                        RAW_MEMORY_DRIVE_STEP_MAX + 0.5,
                        drive_values[0] - 0.5,
                        drive_values[-1] + 0.5,
                    ),
                )
            else:
                axis.text(0.5, 0.5, "no data", ha="center", va="center", transform=axis.transAxes, fontsize=8)

            if row_index == 0:
                axis.set_title(_diagnostic_probability_label(rand_prob), fontsize=10, fontweight="bold")
            if column_index == 0:
                axis.set_ylabel(
                    _diagnostic_level_label(level),
                    fontsize=10,
                    rotation=0,
                    labelpad=62,
                    va="center",
                    fontweight="bold",
                )
            axis.set_xlim(0, RAW_MEMORY_DRIVE_STEP_MAX)
            axis.set_xticks(RAW_MEMORY_DRIVE_STEP_TICKS)
            axis.set_ylim(drive_values[0] - 0.5, drive_values[-1] + 0.5)
            axis.set_yticks(drive_values)
            axis.grid(False)
            axis.tick_params(axis="both", labelsize=7)

    figure.subplots_adjust(left=0.16, right=0.87, bottom=0.08, top=0.94, wspace=0.18, hspace=0.24)
    colorbar_axis = figure.add_axes([0.90, 0.18, 0.018, 0.66])
    scalar_mappable = plt.cm.ScalarMappable(norm=normalizer, cmap=color_map)
    scalar_mappable.set_array([])
    colorbar = figure.colorbar(
        scalar_mappable,
        cax=colorbar_axis,
        ticks=fraction_ticks,
    )
    colorbar.set_label("Fraction of seeds", fontsize=9)
    colorbar.ax.set_yticklabels([f"{value:.1f}" for value in fraction_ticks])
    colorbar.ax.tick_params(labelsize=8)

    figure.supxlabel("Training step", y=0.035, fontsize=12)
    figure.supylabel("Memory drive", x=0.095, fontsize=12)
    figure.suptitle(figure_title, y=0.985, fontsize=11)

    output_path = DIAGNOSTIC_OUTPUT_DIRECTORY / output_filename
    figure.savefig(output_path, bbox_inches="tight", facecolor="white", edgecolor="white")
    plt.close(figure)
    _display_saved_png(output_path)
    print(f"Saved unbinned raw memory drive stacked area plot: {output_path}")
    return [output_path]


def _pooled_age_cdf(frame: pl.DataFrame, distribution_column: str) -> pl.DataFrame:
    age_column = f"{distribution_column}_age"
    age_counts = (
        frame
        .filter((pl.col("step") % STEP_BIN_SIZE) == 0)
        .with_columns(pl.col("step").cast(pl.Int64).alias("step_bin"))
        .select([
            *DIAGNOSTIC_GROUP_COLUMNS,
            "seed",
            "step_bin",
            distribution_column,
        ])
        .explode(distribution_column)
        .rename({distribution_column: age_column})
        .filter(pl.col(age_column).is_not_null())
        .with_columns(((pl.col(age_column) // AGE_BIN_SIZE) * AGE_BIN_SIZE).cast(pl.Int64).alias("age_bin"))
        .group_by([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin", "age_bin"])
        .agg([
            pl.len().alias("age_count"),
            pl.col("seed").n_unique().alias("n_seeds"),
        ])
        .sort([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin", "age_bin"])
        .with_columns(
            pl.col("age_count")
            .sum()
            .over([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
            .alias("age_total")
        )
        .with_columns(
            pl.col("age_count")
            .cum_sum()
            .over([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
            .alias("age_cumulative_count")
        )
        .with_columns(
            pl.when(pl.col("age_total") > 0)
            .then(pl.col("age_cumulative_count") / pl.col("age_total"))
            .otherwise(0.0)
            .alias("age_cdf")
        )
    )
    return age_counts.sort([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin", "age_bin"])


def _age_summary_by_step(
    frame: pl.DataFrame,
    distribution_column: str,
    metric_prefix: str,
) -> pl.DataFrame:
    age_column = f"{distribution_column}_age"
    mean_column = f"{metric_prefix}_age_mean"
    std_column = f"{metric_prefix}_age_std"
    per_run = (
        frame
        .filter((pl.col("step") % STEP_BIN_SIZE) == 0)
        .with_columns(pl.col("step").cast(pl.Int64).alias("step_bin"))
        .select([
            *DIAGNOSTIC_GROUP_COLUMNS,
            "seed",
            "run_id",
            "step_bin",
            distribution_column,
        ])
        .explode(distribution_column)
        .rename({distribution_column: age_column})
        .filter(pl.col(age_column).is_not_null())
        .group_by([*DIAGNOSTIC_GROUP_COLUMNS, "seed", "run_id", "step_bin"])
        .agg([
            pl.col(age_column).mean().alias(mean_column),
            pl.col(age_column).std().fill_null(0.0).alias(std_column),
        ])
    )

    return (
        per_run
        .group_by([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
        .agg([
            pl.col(mean_column).mean().alias(f"{mean_column}_mean"),
            pl.col(std_column).mean().alias(f"{mean_column}_sd"),
            pl.col("seed").n_unique().alias("n_seeds"),
        ])
        .sort([*DIAGNOSTIC_GROUP_COLUMNS, "step_bin"])
    )


def _age_distribution_matrix(
    frame: pl.DataFrame,
    *,
    level: int,
    weight_memory: float,
    rand_prob: float,
    valid_only: bool,
) -> tuple[np.ndarray, list[int], list[int]]:
    subset = (
        frame
        .filter(
            (pl.col("level") == level)
            & (pl.col("weight_memory") == weight_memory)
            & (pl.col("rand_prob") == rand_prob)
            & (pl.col("mercury_split_sensory_raw_latent_valid") == valid_only)
        )
        .select(["step_bin", "age_bin", "age_cdf"])
        .sort(["age_bin", "step_bin"])
    )
    if subset.is_empty():
        return np.empty((0, 0)), [], []

    step_bins = sorted(subset.get_column("step_bin").unique().to_list())
    max_age_bin = int(subset.get_column("age_bin").max() or 0)
    age_bins = list(range(0, max_age_bin + AGE_BIN_SIZE, AGE_BIN_SIZE))
    step_index = {int(step_bin): index for index, step_bin in enumerate(step_bins)}
    age_index = {int(age_bin): index for index, age_bin in enumerate(age_bins)}
    matrix = np.zeros((len(age_bins), len(step_bins)), dtype=np.float32)
    for row in subset.iter_rows(named=True):
        matrix[age_index[int(row["age_bin"])], step_index[int(row["step_bin"])] ] = float(row["age_cdf"])
    matrix = np.maximum.accumulate(matrix, axis=0)
    impossible_age_mask = np.array(age_bins, dtype=float)[:, None] > np.array(step_bins, dtype=float)[None, :]
    matrix[impossible_age_mask] = np.nan
    return matrix, age_bins, step_bins


def plot_age_distribution_heatmaps(
    aggregated_frame: pl.DataFrame,
    *,
    figure_title: str,
    output_filename_prefix: str,
    colorbar_label: str,
    cmap: str = "viridis",
) -> list[Path]:
    panel_specs = [(rand_prob, valid_only) for rand_prob in DIAGNOSTIC_RAND_PROBS for valid_only in DIAGNOSTIC_VALID_ONLY_OPTIONS]
    heatmap_cmap = plt.get_cmap(cmap).copy()
    heatmap_cmap.set_bad(color="white", alpha=0.0)
    output_paths: list[Path] = []

    for level in DIAGNOSTIC_LEVELS:
        figure, axes = plt.subplots(
            nrows=len(DIAGNOSTIC_MEMORY_WEIGHTS),
            ncols=len(panel_specs),
            figsize=(max(9.6, len(panel_specs) * 2.4), max(4.0, len(DIAGNOSTIC_MEMORY_WEIGHTS) * 2.1 + 1.0)),
            squeeze=False,
            dpi=150,
        )
        image = None

        for row_index, weight_memory in enumerate(DIAGNOSTIC_MEMORY_WEIGHTS):
            for column_index, (rand_prob, valid_only) in enumerate(panel_specs):
                axis = axes[row_index, column_index]
                matrix, age_bins, step_bins = _age_distribution_matrix(
                    aggregated_frame,
                    level=level,
                    weight_memory=weight_memory,
                    rand_prob=rand_prob,
                    valid_only=valid_only,
                )
                if matrix.size:
                    image = axis.imshow(
                        matrix,
                        aspect="auto",
                        origin="lower",
                        interpolation="nearest",
                        cmap=heatmap_cmap,
                        vmin=0.0,
                        vmax=1.0,
                    )
                    x_tick_positions = np.linspace(0, len(step_bins) - 1, min(5, len(step_bins)), dtype=int)
                    y_tick_positions = np.linspace(0, len(age_bins) - 1, min(5, len(age_bins)), dtype=int)
                    axis.set_xticks(x_tick_positions, [str(step_bins[index]) for index in x_tick_positions], rotation=45, ha="right")
                    axis.set_yticks(y_tick_positions, [str(age_bins[index]) for index in y_tick_positions])
                else:
                    axis.text(0.5, 0.5, "no data", ha="center", va="center", transform=axis.transAxes)
                if row_index == 0:
                    axis.set_title(f"p={rand_prob:g} valid={valid_only}", fontsize=8)
                if column_index == 0:
                    axis.set_ylabel(f"w={weight_memory:g}", fontsize=8)
                axis.tick_params(axis="both", labelsize=7)

        figure.subplots_adjust(left=0.07, right=0.87, bottom=0.12, top=0.90, wspace=0.30, hspace=0.34)
        if image is not None:
            colorbar_axis = figure.add_axes([0.90, 0.18, 0.018, 0.66])
            colorbar = figure.colorbar(image, cax=colorbar_axis)
            colorbar.set_label(colorbar_label, fontsize=9)
            colorbar.ax.tick_params(labelsize=8)

        figure.supxlabel(f"training step ({STEP_BIN_SIZE}-step sampling)", y=0.04, fontsize=9)
        figure.supylabel(f"age ({AGE_BIN_SIZE}-step bins)", x=0.01, fontsize=9)
        figure.suptitle(f"{figure_title} | L{level}", y=0.985, fontsize=11)

        output_path = DIAGNOSTIC_OUTPUT_DIRECTORY / f"{output_filename_prefix}_level{level}.png"
        figure.savefig(output_path, bbox_inches="tight", facecolor="white", edgecolor="white")
        plt.close(figure)
        _display_saved_png(output_path)
        print(f"Saved age CDF heatmap: {output_path}")
        output_paths.append(output_path)

    return output_paths



states_with_metadata = dataset.states_with_run_metadata(DIAGNOSTIC_METADATA_COLUMNS)
attribution_with_metadata = dataset.attribution_with_run_metadata(DIAGNOSTIC_METADATA_COLUMNS)
history_with_metadata = dataset.history_with_run_metadata(DIAGNOSTIC_METADATA_COLUMNS)

raw_memory_drive_sources = {
    spec["output_filename"]: raw_memory_drive_source_frame(attribution_with_metadata, valid_only=spec["valid_only"])
    for spec in RAW_MEMORY_DRIVE_PLOT_SPECS
}
aggregated_attribution_shares = _aggregate_by_step(attribution_with_metadata, ATTRIBUTION_SHARE_COLUMNS, reducer="mean")
aggregated_attribution_support = _aggregate_by_step(attribution_with_metadata, ATTRIBUTION_SUPPORT_COLUMNS, reducer="mean")
pooled_node_age_cdf = _pooled_age_cdf(states_with_metadata, NODE_AGE_DISTRIBUTION_COLUMN)
pooled_edge_age_cdf = _pooled_age_cdf(states_with_metadata, EDGE_AGE_DISTRIBUTION_COLUMN)
aggregated_node_age_summary = _age_summary_by_step(states_with_metadata, NODE_AGE_DISTRIBUTION_COLUMN, "node")
aggregated_edge_age_summary = _age_summary_by_step(states_with_metadata, EDGE_AGE_DISTRIBUTION_COLUMN, "edge")
aggregated_events = _aggregate_by_step(states_with_metadata, EVENT_COUNT_COLUMNS, reducer="sum")
aggregated_cumulative_events = _aggregate_by_step(states_with_metadata, EVENT_CUMULATIVE_COLUMNS, reducer="max")
aggregated_node_efficiency = _node_efficiency_ratio_by_step(history_with_metadata)

for output_filename, source_frame in raw_memory_drive_sources.items():
    print(f"Raw memory drive source shape ({output_filename}):", source_frame.shape)
print("Attribution share shape:", aggregated_attribution_shares.shape)
print("Pooled node age CDF shape:", pooled_node_age_cdf.shape)
print("Pooled edge age CDF shape:", pooled_edge_age_cdf.shape)
print("Node age summary shape:", aggregated_node_age_summary.shape)
print("Edge age summary shape:", aggregated_edge_age_summary.shape)
print("Events shape:", aggregated_events.shape)
print("Node efficiency shape:", aggregated_node_efficiency.shape)

assert sorted(aggregated_attribution_shares.get_column("weight_memory").unique().to_list()) == DIAGNOSTIC_MEMORY_WEIGHTS
assert int(aggregated_attribution_shares.get_column("n_seeds").min()) == 10
assert float(pooled_node_age_cdf.get_column("age_cdf").min()) >= 0.0
assert float(pooled_node_age_cdf.get_column("age_cdf").max()) <= 1.0
assert float(pooled_edge_age_cdf.get_column("age_cdf").min()) >= 0.0
assert float(pooled_edge_age_cdf.get_column("age_cdf").max()) <= 1.0
assert int(aggregated_node_age_summary.get_column("n_seeds").min()) == 10
assert int(aggregated_edge_age_summary.get_column("n_seeds").min()) == 10
assert int(aggregated_node_efficiency.get_column("n_seeds").min()) == 10

mercury_diagnostic_plot_paths = [
    *plot_metric_lines(
        aggregated_attribution_shares,
        ATTRIBUTION_SHARE_COLUMNS,
        figure_title="Mercury BMU Drive Shares",
        output_filename_prefix=f"mercury_drive_shares_bin{STEP_BIN_SIZE}",
        y_axis_label="share",
        y_limits=(0.0, 1.0),
    ),
    *(
        path
        for spec in RAW_MEMORY_DRIVE_PLOT_SPECS
        for path in plot_raw_memory_drive_stacked_area(
            raw_memory_drive_sources[spec["output_filename"]],
            figure_title=spec["title"],
            output_filename=spec["output_filename"],
        )
    ),
    *plot_metric_lines(
        aggregated_attribution_support,
        ATTRIBUTION_SUPPORT_COLUMNS,
        figure_title="Mercury BMU Support Totals",
        output_filename_prefix=f"mercury_support_totals_bin{STEP_BIN_SIZE}",
        y_axis_label="support",
    ),
    *plot_metric_lines(
        aggregated_node_efficiency,
        ["node_efficiency_ratio"],
        figure_title="Mercury Node Efficiency Ratio",
        output_filename_prefix=f"mercury_node_efficiency_ratio_bin{STEP_BIN_SIZE}",
        y_axis_label="latent states / ground-truth state",
        show_sd_band=True,
    ),
    *plot_age_distribution_heatmaps(
        pooled_node_age_cdf,
        figure_title="Mercury Node Age CDF",
        output_filename_prefix=f"mercury_node_age_cdf_heatmap_bin{STEP_BIN_SIZE}_agebin{AGE_BIN_SIZE}",
        colorbar_label="node age CDF",
    ),
    *plot_age_distribution_heatmaps(
        pooled_edge_age_cdf,
        figure_title="Mercury Edge Age CDF",
        output_filename_prefix=f"mercury_edge_age_cdf_heatmap_bin{STEP_BIN_SIZE}_agebin{AGE_BIN_SIZE}",
        colorbar_label="edge age CDF",
    ),
    *plot_metric_lines(
        aggregated_node_age_summary,
        ["node_age_mean"],
        figure_title="Mercury Node Age Mean +/- Distribution SD",
        output_filename_prefix=f"mercury_node_age_mean_distribution_sd_bin{STEP_BIN_SIZE}",
        y_axis_label="age in steps",
        show_sd_band=True,
        band_floor=0.0,
    ),
    *plot_metric_lines(
        aggregated_edge_age_summary,
        ["edge_age_mean"],
        figure_title="Mercury Edge Age Mean +/- Distribution SD",
        output_filename_prefix=f"mercury_edge_age_mean_distribution_sd_bin{STEP_BIN_SIZE}",
        y_axis_label="age in steps",
        show_sd_band=True,
        band_floor=0.0,
    ),
    *plot_metric_lines(
        aggregated_events,
        EVENT_COUNT_COLUMNS,
        figure_title="Mercury Clone and Prune Events",
        output_filename_prefix=f"mercury_clone_prune_events_bin{STEP_BIN_SIZE}",
        y_axis_label="event count per bin",
        show_sd_band=True,
    ),
    *plot_metric_lines(
        aggregated_cumulative_events,
        EVENT_CUMULATIVE_COLUMNS,
        figure_title="Mercury Cumulative Clone and Prune Events",
        output_filename_prefix=f"mercury_clone_prune_cumulative_bin{STEP_BIN_SIZE}",
        y_axis_label="cumulative count",
        show_sd_band=True,
    ),
]

print("Generated Mercury diagnostic plots:")
for path in mercury_diagnostic_plot_paths:
    print(path)
